# AQG Dataset Pipeline — Final Checkpoint

Notebook ini menjalankan pipeline end-to-end dari materi Markdown ke dataset JSONL.
Mulai dari 2 modul dulu (`01-Berkenalan-dengan-python` dan `02-berinteraksi-dengan-data`),
lalu bisa diperluas ke semua modul.

**Alur:**
1. Chunking materi → inspeksi hasil
2. Prompt construction → inspeksi contoh prompt
3. Generate 1 soal sample (test LLM connection)
4. Jalankan pipeline untuk 2 modul
5. Verifikasi output JSONL dengan HuggingFace datasets
6. Jalankan pipeline untuk semua modul (opsional)

> **Catatan Gemini API:**
> - Model: `gemini-2.5-flash` via `langchain-google-genai`
> - API key disimpan di `.env` sebagai `GOOGLE_API_KEY`
> - Jika terjadi error, pipeline **menyimpan data yang sudah berhasil** ke `accumulated.jsonl`
> - Jalankan ulang — pipeline akan melanjutkan dari checkpoint, data lama tidak hilang


In [1]:
import sys, os
from pathlib import Path

# Root project = 2 level di atas src/pipeline/
project_root = Path(os.getcwd())
if project_root.name == 'pipeline':
    project_root = project_root.parent.parent
elif project_root.name == 'src':
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv()

print(f'Working directory: {os.getcwd()}')
print('Environment loaded.')


Working directory: d:\2-Project\AQG
Environment loaded.


## 1. Chunking — Inspeksi Hasil

In [2]:
from src.dataset.chunker import chunk_markdown, chunk_all_materials

# Test dengan 1 file dulu
chunks = chunk_markdown('dataset_aqg/materi/01-Berkenalan-dengan-python/01-perkenalan-pythn.md')
print(f'Total chunks: {len(chunks)}')
print()
for i, c in enumerate(chunks):
    print(f'[{i}] tokens={c.token_count:3d} | has_code={c.has_code} | heading="{c.section_heading[:50]}"')

Total chunks: 7

[0] tokens=245 | has_code=True | heading="# Pengenalan Python"
[1] tokens=113 | has_code=False | heading="### Python 2.x"
[2] tokens=179 | has_code=False | heading="### Python 3.x"
[3] tokens= 94 | has_code=False | heading="### Python Software Foundation (PSF)"
[4] tokens= 55 | has_code=False | heading="### Python Enhancement Proposals (PEP)"
[5] tokens=309 | has_code=False | heading="### Zen of Python (PEP 20)"
[6] tokens=169 | has_code=False | heading="## Mengapa Python?"


In [3]:
# Lihat isi chunk pertama
print('=== CHUNK 0 ===')
print(chunks[0].text[:500])

=== CHUNK 0 ===
# Pengenalan Python

Halo, calon programmer! Selamat datang pada modul pertama kelas **Memulai Pemrograman dengan Python**. Sebelum Anda mulai membuat program, mari mulai pembelajaran dengan mengenal terlebih dahulu bahasa pemrograman Python. Python adalah bahasa pemrograman multifungsi yang dirilis pada tahun **1991** oleh **Guido van Rossum (GvR)**. Beliau membuat Python sebagai bahasa pemrograman yang mudah dibaca dan dimengerti (*readable*) serta memiliki kemampuan penanganan kesalahan (*exc


In [4]:
# Chunk semua file di modul 01
all_chunks_01 = chunk_all_materials('dataset_aqg/materi/01-Berkenalan-dengan-python')
all_chunks_02 = chunk_all_materials('dataset_aqg/materi/02-berinteraksi-dengan-data')

print(f'Modul 01: {len(all_chunks_01)} chunks')
print(f'Modul 02: {len(all_chunks_02)} chunks')
print(f'Total: {len(all_chunks_01) + len(all_chunks_02)} chunks')

Modul 01: 38 chunks
Modul 02: 34 chunks
Total: 72 chunks


## 2. Prompt Construction — Inspeksi Contoh Prompt

In [5]:
from src.dataset.prompt_constructor import TaskParams, build_prompt
from src.dataset.concept_list import get_concepts_for_module

# Ambil chunk dengan code block
code_chunks = [c for c in all_chunks_01 if c.has_code]
sample_chunk = code_chunks[0] if code_chunks else all_chunks_01[0]

params = TaskParams(concept='Variable dan Assignment', difficulty='medium', question_type='MCQ')
prompt_input = build_prompt(sample_chunk, params)

print('=== INPUT PROMPT ===')
print(prompt_input.input)
print()
print(f'Token count chunk: {sample_chunk.token_count}')
print(f'Has code: {sample_chunk.has_code}')

=== INPUT PROMPT ===
Konteks: # Pengenalan Python

Halo, calon programmer! Selamat datang pada modul pertama kelas **Memulai Pemrograman dengan Python**. Sebelum Anda mulai membuat program, mari mulai pembelajaran dengan mengenal terlebih dahulu bahasa pemrograman Python. Python adalah bahasa pemrograman multifungsi yang dirilis pada tahun **1991** oleh **Guido van Rossum (GvR)**. Beliau membuat Python sebagai bahasa pemrograman yang mudah dibaca dan dimengerti (*readable*) serta memiliki kemampuan penanganan kesalahan (*exception handling*). Berdasarkan tujuan tersebut, Guido van Rossum berhasil menjadikan Python sebagai bahasa pemrograman yang dapat diimplementasikan ke dalam berbagai sektor. Python dapat digunakan untuk membangun website (server-side), analisis data, hingga pembelajaran mesin (*machine learning*). Python memiliki ciri khas tersendiri sebagai salah satu pemrograman populer. Salah satu ciri khas yang paling dikenal adalah Python tidak mewajibkan penggunaan titik koma 

## 3. Test LLM Connection — Generate 1 Soal Sample

In [6]:
from src.dataset.synthetic_generator import _build_llm_client, generate_datapoint

llm = _build_llm_client()
print('LLM client created.')

# Generate 1 soal sample
result = generate_datapoint(prompt_input, llm, max_retries=2)

if result:
    print('\n=== GENERATED DATA POINT ===')
    print(f'TARGET:\n{result.target}')
    print(f'\nMETADATA: {result.metadata}')
else:
    print('GAGAL generate — cek API key dan koneksi')

[DEBUG] model=glm-4.7, base_url=https://api.z.ai/api/paas/v4, api_key=SET
LLM client created.

=== GENERATED DATA POINT ===
TARGET:
Pertanyaan: Perhatikan kode berikut: `nilai = 85` diikuti dengan `nilai = "Lulus"`. Apa yang terjadi pada variabel `nilai`? Jawaban benar: Variabel `nilai` akan menyimpan teks "Lulus" dan tipe datanya berubah otomatis menjadi string. Distraktor: 1) Program error karena tipe data variabel tidak boleh diubah setelah diberi nilai awal. 2) Variabel `nilai` akan menyimpan gabungan 85 dan "Lulus". 3) Python akan mengabaikan baris kedua sehingga `nilai` tetap bernilai 85. 4) Variabel `nilai` harus dideklarasikan ulang dengan perintah `new` sebelum bisa diubah nilainya.

METADATA: {'difficulty': 'medium', 'question_type': 'MCQ', 'concept': 'Variable dan Assignment', 'misconception_tags': [], 'source_file': 'dataset_aqg\\materi\\01-Berkenalan-dengan-python\\01-perkenalan-pythn.md', 'section': '# Pengenalan Python', 'source': 'synthetic', 'validated': False}


## 4. Jalankan Pipeline untuk 2 Modul

In [7]:
from src.pipeline.dataset_pipeline import run_pipeline
import shutil
from pathlib import Path

output_dir = 'dataset_aqg/output_modul'

# JANGAN hapus output_dir jika ingin melanjutkan dari checkpoint!
# Hapus hanya jika ingin mulai dari awal:
# if Path(output_dir).exists():
#     shutil.rmtree(output_dir)
#     print(f'Output lama dihapus: {output_dir}')

accumulated = Path(output_dir) / 'accumulated.jsonl'
if accumulated.exists():
    import json
    count = sum(1 for _ in open(accumulated, encoding='utf-8'))
    print(f'[INFO] Melanjutkan — {count} data point sudah tersimpan di accumulated.jsonl')
else:
    print('[INFO] Mulai dari awal.')

print('Menjalankan pipeline untuk modul 01...')

[INFO] Melanjutkan — 18 data point sudah tersimpan di accumulated.jsonl
Menjalankan pipeline untuk modul 01...


In [ ]:
# Modul 01
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    max_per_chunk=2,
    section_filter='01-Berkenalan-dengan-python',
    difficulties=['easy', 'medium'],
    question_types=['MCQ'],
)

[DEBUG] model=glm-4.7, base_url=https://api.z.ai/api/paas/v4, api_key=SET

[SECTION] 01-Berkenalan-dengan-python
[INFO] 38 chunks dari 7 file
[INFO] 11 konsep: ['Sejarah Python', 'Ciri Khas Python', 'Versi Python 2.x']...
[INFO] 76 prompt inputs akan di-generate


Generating 01-Berkenalan-dengan-python:  16%|█▌        | 12/76 [07:20<35:44, 33.50s/soal]

In [ ]:
# Modul 02
run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=output_dir,
    max_per_chunk=2,
    section_filter='02-berinteraksi-dengan-data',
    difficulties=['easy', 'medium'],
    question_types=['MCQ'],
)

## 5. Verifikasi Output JSONL

In [ ]:
import json
from pathlib import Path

# Cek file yang dihasilkan
output_path = Path(output_dir)
for f in sorted(output_path.iterdir()):
    size = f.stat().st_size
    print(f'{f.name:30s} {size:8d} bytes')

In [ ]:
# Baca dataset_info.json
with open(output_path / 'dataset_info.json', encoding='utf-8') as f:
    info = json.load(f)

print('=== DATASET INFO ===')
print(json.dumps(info, indent=2, ensure_ascii=False))

In [ ]:
# Inspeksi 3 contoh dari train.jsonl
train_file = output_path / 'train.jsonl'
samples = []
with open(train_file, encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        samples.append(json.loads(line))

for i, s in enumerate(samples):
    print(f'\n=== SAMPLE {i+1} ===')
    print(f'CONCEPT: {s["metadata"]["concept"]}')
    print(f'DIFFICULTY: {s["metadata"]["difficulty"]}')
    print(f'TARGET:\n{s["target"]}')

In [ ]:
# Verifikasi dengan HuggingFace datasets
try:
    from datasets import load_dataset
    ds = load_dataset('json', data_files={
        'train': str(output_path / 'train.jsonl'),
        'validation': str(output_path / 'validation.jsonl'),
        'test': str(output_path / 'test.jsonl'),
    })
    print('=== HUGGINGFACE DATASET ===')
    print(ds)
    print(f'\nContoh dari train split:')
    print(ds['train'][0])
except ImportError:
    print('datasets library tidak tersedia — skip HuggingFace verification')

## 6. Jalankan Pipeline untuk Semua Modul (Opsional)

Uncomment cell di bawah untuk menjalankan pipeline pada semua 11 modul.
Estimasi waktu: ~30-60 menit tergantung rate limit OpenRouter.

In [ ]:
# # Uncomment untuk jalankan semua modul
# import shutil
# 
# output_full = 'dataset_aqg/output_full'
# if Path(output_full).exists():
#     shutil.rmtree(output_full)
# 
# run_pipeline(
#     materi_dir='dataset_aqg/materi',
#     output_dir=output_full,
#     max_per_chunk=3,
#     difficulties=['easy', 'medium', 'hard'],
#     question_types=['MCQ'],
# )
print('Cell ini dinonaktifkan. Uncomment untuk jalankan semua modul.')

## 7. Dry Run — Test Pipeline Tanpa LLM

In [ ]:
# Dry run untuk verifikasi pipeline tanpa memanggil LLM
import shutil

dry_output = 'dataset_aqg/output_dryrun'
if Path(dry_output).exists():
    shutil.rmtree(dry_output)

run_pipeline(
    materi_dir='dataset_aqg/materi',
    output_dir=dry_output,
    max_per_chunk=1,
    section_filter='01-Berkenalan-dengan-python',
    dry_run=True,
)

# Verifikasi output dry run
dry_path = Path(dry_output)
with open(dry_path / 'dataset_info.json', encoding='utf-8') as f:
    dry_info = json.load(f)
print(f'Dry run total: {dry_info["total"]} data points')
print(f'Splits: {dry_info["splits"]}')